In [ ]:
# KAPipe CDR Baseline
# This notebook adapts the published KAPipe CDR pipeline to the thesis evaluation schema.
# Run order: configure paths, validate snapshots and imports, build the pipeline, then run smoke or full evaluation.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# KAPipe source code and pretrained snapshots are kept outside src because they come from related work.
KAPIPE_SOURCE_DIR = PROJECT_ROOT / "reference code from related work" / "kapipe-main" / "kapipe-main"
KAPIPE_MODEL_ROOT = PROJECT_ROOT / "reference code from related work" / "kapipe-model"

# Evaluation parameters.
DATASET_NAME = "ade"  # ADE is mapped to KAPipe's chemical-disease relation format by the adapter.
EVAL_SAMPLE_N = 100  # Small, fixed prefix used only when RUN_SMOKE is enabled.
PRIMARY_METRIC = "anchor_window"  # Matches the main thesis extraction metric for fair comparison.
RETRIEVAL_SIZE = 15  # Retains the retrieval depth used for the reported KAPipe baseline.
DEVICE = "cuda:0"  # The entity and relation models are evaluated on the first CUDA device.
USE_CROSS_ENCODER = True  # Keeps KAPipe's reranking stage; disable only for a dedicated ablation.

# Execution switches are independent so a quick validation never implies a full run.
RUN_SMOKE = False  # Enable first when checking a new environment or snapshot installation.
RUN_FULL_EVAL = True  # Final baseline run over the selected evaluation split.
FULL_EVAL_SAMPLE_N = None  # None loads the complete ADE test set; set an integer for a bounded rerun.

# Output names encode the dataset and sample count to keep smoke and full results separate.
OUTPUT_DIR = PROJECT_ROOT / "results" / "eval_report" / "kapipe_cdr_baseline"
SMOKE_RUN_NAME = f"kapipe-cdr_{DATASET_NAME}-n{EVAL_SAMPLE_N}"
FULL_RUN_NAME = f"kapipe-cdr_{DATASET_NAME}-n{'all' if FULL_EVAL_SAMPLE_N is None else FULL_EVAL_SAMPLE_N}"


In [ ]:
# 1. Validate every external snapshot and Python dependency before allocating GPU memory.
# The preflight dictionary is intentionally displayed so missing components can be corrected in one pass.
import importlib.util

from src.kapipe_cdr_baseline import default_snapshot_paths, validate_snapshot_paths

SNAPSHOT_PATHS = default_snapshot_paths(KAPIPE_MODEL_ROOT)
MISSING_SNAPSHOT_PATHS = validate_snapshot_paths(SNAPSHOT_PATHS)
REQUIRED_IMPORTS = ["faiss", "jsonlines", "numpy", "opt_einsum", "pyhocon", "torch", "tqdm", "transformers"]
MISSING_IMPORTS = [name for name in REQUIRED_IMPORTS if importlib.util.find_spec(name) is None]

preflight = {
    "kapipe_source_dir": str(KAPIPE_SOURCE_DIR),
    "kapipe_model_root": str(KAPIPE_MODEL_ROOT),
    "snapshot_paths": {name: str(path) for name, path in SNAPSHOT_PATHS.as_dict().items()},
    "missing_snapshot_paths": {
        name: [str(path) for path in paths]
        for name, paths in MISSING_SNAPSHOT_PATHS.items()
    },
    "missing_imports": MISSING_IMPORTS,
    "device": DEVICE,
}
preflight


In [ ]:
# 2. Build the KAPipe pipeline only when at least one evaluation switch is enabled.
# Fail-fast checks keep partial snapshot installations from producing incomplete or incomparable outputs.
from src.kapipe_cdr_baseline import build_kapipe_cdr_pipeline, format_missing_snapshot_paths

pipeline = None
if RUN_SMOKE or RUN_FULL_EVAL:
    if MISSING_SNAPSHOT_PATHS:
        raise FileNotFoundError(format_missing_snapshot_paths(MISSING_SNAPSHOT_PATHS))
    if MISSING_IMPORTS:
        raise RuntimeError(
            "Missing KAPipe baseline dependencies: " + ", ".join(MISSING_IMPORTS)
            + ". Install them in the Master_thesis conda environment and retry."
        )
    pipeline = build_kapipe_cdr_pipeline(
        kapipe_source_dir=KAPIPE_SOURCE_DIR,
        snapshot_paths=SNAPSHOT_PATHS,
        device=DEVICE,
        use_cross_encoder=USE_CROSS_ENCODER,
    )
pipeline


In [ ]:
# 3. Optionally run the fixed-size smoke evaluation.
# This uses the same adapter, retrieval depth, metric, and save format as the full evaluation.
from IPython.display import Markdown, display

from src.data_io import load_dataset
from src.kapipe_cdr_baseline import run_kapipe_cdr_baseline, save_run_outputs

smoke_result = None
smoke_saved_paths = None
if RUN_SMOKE:
    smoke_samples = load_dataset(DATASET_NAME, n=EVAL_SAMPLE_N)
    smoke_result = run_kapipe_cdr_baseline(
        samples=smoke_samples,
        pipeline=pipeline,
        dataset=DATASET_NAME,
        retrieval_size=RETRIEVAL_SIZE,
        primary_metric=PRIMARY_METRIC,
    )
    smoke_saved_paths = save_run_outputs(smoke_result, OUTPUT_DIR, SMOKE_RUN_NAME)
    display(Markdown(f"```text\n{smoke_result['formatted_report']}\n```"))

smoke_saved_paths


In [ ]:
# 4. Run the complete evaluation after the preflight or smoke result has been inspected.
# Predictions, converted KAPipe documents, and the formatted metric report are saved together.
full_result = None
full_saved_paths = None
if RUN_FULL_EVAL:
    full_samples = load_dataset(DATASET_NAME, n=FULL_EVAL_SAMPLE_N)
    full_result = run_kapipe_cdr_baseline(
        samples=full_samples,
        pipeline=pipeline,
        dataset=DATASET_NAME,
        retrieval_size=RETRIEVAL_SIZE,
        primary_metric=PRIMARY_METRIC,
    )
    full_saved_paths = save_run_outputs(full_result, OUTPUT_DIR, FULL_RUN_NAME)
    display(Markdown(f"```text\n{full_result['formatted_report']}\n```"))

full_saved_paths
